# Sunflower Leaf Disease Detection — MobileNetV2
## ✅ Fixed: includes 'Other' class to reject non-sunflower images

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
!pip install tensorflow wandb -q

## 3. Imports

In [ ]:
import os
import json
import shutil
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import clear_output

print(f'TensorFlow version: {tf.__version__}')

## 4. Download 'Other' Images (non-sunflower)

Tunahitaji picha za vitu ambavyo **si** majani ya alizeti ili model ijifunze kukataa.
Tunatumia subset ndogo ya ImageNet kutoka TensorFlow Datasets — rahisi na haraka.

In [ ]:
import tensorflow_datasets as tfds

OTHER_DIR = pathlib.Path('/content/other_images')
OTHER_DIR.mkdir(exist_ok=True)

# Load a small subset of imagenet_v2 — these are general images (not sunflower leaves)
# We take only 400 images to balance with sunflower classes
print('Downloading ~400 non-sunflower images from ImageNet subset...')

ds = tfds.load('imagenet_v2', split='test[:5%]', as_supervised=True, shuffle_files=True)

count = 0
MAX_OTHER = 400

for img_tensor, label in ds:
    if count >= MAX_OTHER:
        break
    img_np = img_tensor.numpy()
    # Skip any image that might accidentally be a sunflower (ImageNet class 985)
    if label.numpy() == 985:  # sunflower class in ImageNet
        continue
    from PIL import Image
    import io
    img_pil = Image.fromarray(img_np).convert('RGB').resize((224, 224))
    img_pil.save(OTHER_DIR / f'other_{count:04d}.jpg')
    count += 1

print(f'✅ Saved {count} non-sunflower images to {OTHER_DIR}')

## 5. Build Combined Dataset Directory

Tunaunda folder mpya yenye classes zote 5:
- `Fresh Leaf`
- `Downy mildew`  
- `Gray mold`
- `Leaf scars`
- `Other` ← **mpya** — inakataa vitu visivyo alizeti

In [ ]:
# Original sunflower dataset
ORIGINAL_DIR = pathlib.Path('/content/drive/MyDrive/Original Image/SUN')

# New combined dataset
COMBINED_DIR = pathlib.Path('/content/combined_dataset')

# Clear and recreate
if COMBINED_DIR.exists():
    shutil.rmtree(COMBINED_DIR)
COMBINED_DIR.mkdir()

# Copy original sunflower classes
for class_dir in sorted(ORIGINAL_DIR.iterdir()):
    if class_dir.is_dir():
        dest = COMBINED_DIR / class_dir.name
        shutil.copytree(class_dir, dest)
        images = list(dest.glob('*'))
        print(f'  {class_dir.name}: {len(images)} images')

# Add 'Other' class
other_dest = COMBINED_DIR / 'Other'
shutil.copytree(OTHER_DIR, other_dest)
print(f'  Other: {len(list(other_dest.glob("*")))} images')

print(f'\n✅ Combined dataset ready at {COMBINED_DIR}')
print(f'Classes: {sorted([d.name for d in COMBINED_DIR.iterdir() if d.is_dir()])}')

## 6. Configuration

In [ ]:
train_dir = str(COMBINED_DIR)

SEED       = 12
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 0.01
EARLY_STOPPING_CRITERIA = 3

CLASS_LABELS = sorted(os.listdir(train_dir))
NUM_CLASSES  = len(CLASS_LABELS)

print(f'Classes ({NUM_CLASSES}): {CLASS_LABELS}')
# Should print: ['Downy mildew', 'Fresh Leaf', 'Gray mold', 'Leaf scars', 'Other']

## 7. Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2,
    horizontal_flip  = True,
    vertical_flip    = True,         # helps model learn 'Other' from all angles
    zoom_range       = 0.15,
    rotation_range   = 15,
    brightness_range = [0.8, 1.2],   # phone camera lighting variation
    shear_range      = 0.1,
)
valid_datagen = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2,
)

train_generator = train_datagen.flow_from_directory(
    directory   = train_dir,
    target_size = (IMG_HEIGHT, IMG_WIDTH),
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    subset      = 'training',
    seed        = SEED,
)

valid_generator = valid_datagen.flow_from_directory(
    directory   = train_dir,
    target_size = (IMG_HEIGHT, IMG_WIDTH),
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    subset      = 'validation',
    seed        = SEED,
)

print(f'Training samples  : {train_generator.samples}')
print(f'Validation samples: {valid_generator.samples}')
print(f'Class indices     : {train_generator.class_indices}')

## 8. Class Weights (to handle imbalance)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.unique(train_generator.classes),
    y            = train_generator.classes
)

class_weights = dict(enumerate(class_weights_array))
print('Class weights:', class_weights)
# Higher weight for 'Other' means model is penalized more for misclassifying random images

## 9. Build MobileNetV2 Model (5 classes now)

In [ ]:
def build_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        input_shape = (IMG_HEIGHT, IMG_WIDTH, 3),
        include_top = False,
        weights     = 'imagenet',
        alpha       = 1.0,
    )
    base.trainable = False

    inputs = tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(
            256, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(1e-5))(x)
    x = tf.keras.layers.Dropout(0.4)(x)   # slightly higher dropout for better generalisation
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classification')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer = tf.keras.optimizers.SGD(learning_rate=LR, momentum=0.9),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    return model, base

model, mobilenet_base = build_model(NUM_CLASSES)
clear_output()
model.summary()
print(f'\nOutput classes: {NUM_CLASSES} → {CLASS_LABELS}')

## 10. Train — Phase 1 (frozen backbone)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor              = 'val_loss',
    patience             = EARLY_STOPPING_CRITERIA,
    restore_best_weights = True,
    verbose              = 1,
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 2,
    min_lr   = 1e-6,
    verbose  = 1,
)

history = model.fit(
    x               = train_generator,
    epochs          = EPOCHS,
    validation_data = valid_generator,
    class_weight    = class_weights,   # ← handles class imbalance
    callbacks       = [early_stop, reduce_lr],
)

history_df = pd.DataFrame(history.history)
print('Phase 1 complete.')

## 11. Fine-Tuning — Phase 2 (unfreeze top 30 layers)

In [ ]:
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy'],
)

history_ft = model.fit(
    x               = train_generator,
    epochs          = 10,
    validation_data = valid_generator,
    class_weight    = class_weights,
    callbacks       = [early_stop, reduce_lr],
)
print('Fine-tuning complete.')

## 12. Evaluation

In [ ]:
y_true  = valid_generator.classes
y_probs = model.predict(valid_generator)
y_pred  = np.argmax(y_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_LABELS))

cm_data = confusion_matrix(y_true, y_pred)
cm = pd.DataFrame(cm_data, columns=CLASS_LABELS, index=CLASS_LABELS)
cm.index.name   = 'Actual'
cm.columns.name = 'Predicted'
plt.figure(figsize=(10, 8))
plt.title('Confusion Matrix — MobileNetV2 (5 classes)', fontsize=16)
sns.heatmap(cm, cbar=False, cmap='Blues', annot=True, fmt='g')
plt.show()

# Check how well 'Other' class is rejected
other_idx = CLASS_LABELS.index('Other')
other_true = (y_true == other_idx)
other_pred = (y_pred == other_idx)
other_precision = (other_true & other_pred).sum() / other_pred.sum() if other_pred.sum() > 0 else 0
other_recall    = (other_true & other_pred).sum() / other_true.sum() if other_true.sum() > 0 else 0
print(f"\n'Other' class precision : {other_precision:.1%}")
print(f"'Other' class recall    : {other_recall:.1%}")
print("High recall = model correctly rejects non-sunflower images")

## 13. Save Model (.h5)

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/mobilenetv2-plant-v2.h5'
model.save(MODEL_SAVE_PATH)
print(f'✅ Model saved to {MODEL_SAVE_PATH}')
print(f'   Classes : {CLASS_LABELS}')
print(f'   Input   : {model.input_shape}')
print(f'   Output  : {model.output_shape}  ← now {NUM_CLASSES} classes')

## 14. Update Django views.py CLASS_NAMES

**MUHIMU**: Baada ya retrain, `CLASS_NAMES` kwenye `views.py` lazima ilingane na order ya training.
Run cell hii uone order sahihi.

In [ ]:
print('='*50)
print('COPY HII KWENYE views.py → CLASS_NAMES:')
print('='*50)
print()
print('CLASS_NAMES = [')
for i, name in enumerate(CLASS_LABELS):
    comma = ',' if i < len(CLASS_LABELS)-1 else ''
    print(f'    "{name}"{comma}')
print(']')
print()
print('Class index mapping:')
for k, v in train_generator.class_indices.items():
    print(f'  {v}: {k}')

## 15. Convert to TFLite (optional)

In [ ]:
TFLITE_DIR = pathlib.Path('tflite_models')
TFLITE_DIR.mkdir(exist_ok=True)

# Float32
conv_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = conv_f32.convert()
tflite_path = TFLITE_DIR / 'mobilenetv2_plant_v2_f32.tflite'
tflite_path.write_bytes(tflite_model)
print(f'Saved {tflite_path.name}  ({tflite_path.stat().st_size/1e6:.2f} MB)')

# Save class labels
labels_path = TFLITE_DIR / 'class_labels.json'
with open(labels_path, 'w') as f:
    json.dump(CLASS_LABELS, f, indent=2)
print(f'Class labels → {labels_path}')
print(json.dumps(CLASS_LABELS, indent=2))

## 16. Test: picha ya kitu kisichokuwa alizeti

In [ ]:
from PIL import Image
import io
from google.colab import files

def predict_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB').resize((224, 224))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=0)
    probs = model.predict(arr)[0]

    print('\nAll class probabilities:')
    for name, prob in zip(CLASS_LABELS, probs):
        bar = '█' * int(prob * 40)
        print(f'  {name:<20} {prob*100:5.1f}% {bar}')

    pred_class = CLASS_LABELS[np.argmax(probs)]
    confidence = float(np.max(probs))

    print(f'\n→ Predicted : {pred_class}')
    print(f'→ Confidence: {confidence*100:.1f}%')

    if pred_class == 'Other' or confidence < 0.65:
        print('✅ CORRECTLY REJECTED — not a sunflower leaf')
    else:
        print(f'✅ ACCEPTED — {pred_class}')

print('Upload a test image (can be anything — phone, food, random object):')
uploaded = files.upload()
for fname, img_bytes in uploaded.items():
    print(f'\n─── {fname} ───')
    predict_image(img_bytes)

## 17. Download Model


In [ ]:
from google.colab import files
import zipfile

zip_path = 'mobilenetv2_v2_bundle.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in TFLITE_DIR.iterdir():
        zf.write(f, f.name)

print(f'Bundle ready: {zip_path}')
files.download(zip_path)
files.download(MODEL_SAVE_PATH)